# 01 — BDD100K Dataset Setup & Download

**Goal:** Identify the legitimate BDD100K source, explain what to download, and verify files.

## BDD100K Overview

| Item | Detail |
|---|---|
| **Full name** | Berkeley DeepDrive 100K |
| **Official site** | [bdd-data.berkeley.edu](https://bdd-data.berkeley.edu) |
| **Images** | 100,000 driving images (1280×720) |
| **Split** | 70K train / 10K val / 20K test |
| **Detection classes** | 10: person, rider, car, truck, bus, train, motorcycle, bicycle, traffic light, traffic sign |
| **License** | BSD 3-Clause (research use) |

## What to Download

### Required NOW (object detection)
| File | Description | Size |
|---|---|---|
| `bdd100k_images_100k.zip` | All 100K images | ~6.4 GB |
| `bdd100k_labels_release.zip` | Detection labels (JSON) | ~100 MB |

### Useful LATER (lane marking — do NOT train on yet)
| File | Description |
|---|---|
| `bdd100k_lane_marks_*.json` | Lane polyline annotations |
| `bdd100k_drivable_maps.zip` | Drivable area segmentation masks |

> 💡 Download the lane/drivable files now if bandwidth allows — they'll be used in future notebooks.

## 1 — Install Dependencies

In [ ]:
!pip install -q gdown tqdm pyyaml

## 2 — Mount Google Drive

BDD100K is large (~6.5 GB images). **Store it on Google Drive** so you don't re-download each Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 — Path Configuration

In [ ]:
import os

# ── Configure these paths ──────────────────────────────────────────
# Root directory on Google Drive for all EcoCAR data
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"

# Where BDD100K raw data will be stored
BDD_RAW_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_raw")

# Subdirectories
BDD_IMAGES_DIR = os.path.join(BDD_RAW_DIR, "images", "100k")
BDD_LABELS_DIR = os.path.join(BDD_RAW_DIR, "labels")

# Create directories
for d in [BDD_RAW_DIR, BDD_IMAGES_DIR, BDD_LABELS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"📁 {d}")

print(f"\n✅ Directory structure ready")

## 4 — Download Instructions

### Option A: Official website (recommended)

1. Go to **[bdd-data.berkeley.edu](https://bdd-data.berkeley.edu)**
2. Create a free account / sign in
3. Download:
   - **Images**: `bdd100k_images_100k.zip`
   - **Labels**: `bdd100k_labels_release.zip`
4. Upload the zip files to your Google Drive at the path below
5. Run the extraction cells in this notebook

### Option B: Hugging Face mirror

If the official site is unavailable, BDD100K is also hosted on Hugging Face:
- Dataset: `dgural/bdd100k` or search "BDD100K" on [huggingface.co/datasets](https://huggingface.co/datasets)

### Option C: Upload directly to Colab

If you have the files locally, upload them to Colab:
```python
from google.colab import files
uploaded = files.upload()  # Select your zip files
```

## 5 — Extract Downloaded Files

Once you've uploaded the zip files to your Drive, run the cells below.

In [ ]:
import zipfile

# ── Set paths to your downloaded zip files ──────────────────────────
# Adjust these if you placed the zips elsewhere on Drive
IMAGES_ZIP = os.path.join(ECOCAR_ROOT, "downloads", "bdd100k_images_100k.zip")
LABELS_ZIP = os.path.join(ECOCAR_ROOT, "downloads", "bdd100k_labels_release.zip")

def extract_if_needed(zip_path, extract_to):
    """Extract a zip file if it exists and target seems empty."""
    if not os.path.isfile(zip_path):
        print(f"⚠️ Zip not found: {zip_path}")
        print(f"   Please download and upload to this path first.")
        return False
    
    print(f"📦 Extracting: {os.path.basename(zip_path)}")
    print(f"   To: {extract_to}")
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_to)
    
    print(f"   ✅ Done!")
    return True

In [ ]:
# Extract images
extract_if_needed(IMAGES_ZIP, BDD_RAW_DIR)

In [ ]:
# Extract labels
extract_if_needed(LABELS_ZIP, BDD_RAW_DIR)

## 6 — Verify Files Exist

In [ ]:
def check_path(path, description):
    """Check if a path exists and report."""
    if os.path.exists(path):
        if os.path.isdir(path):
            count = len(os.listdir(path))
            print(f"✅ {description}: {path} ({count} items)")
        else:
            size_mb = os.path.getsize(path) / (1024**2)
            print(f"✅ {description}: {path} ({size_mb:.1f} MB)")
        return True
    else:
        print(f"❌ {description}: NOT FOUND at {path}")
        return False

print("\n" + "="*60)
print(" FILE VERIFICATION")
print("="*60)

# Expected paths after extraction
# BDD100K typically extracts to: bdd100k/images/100k/train/, .../val/
#                                bdd100k/labels/
checks = {
    "Train images": os.path.join(BDD_RAW_DIR, "bdd100k", "images", "100k", "train"),
    "Val images":   os.path.join(BDD_RAW_DIR, "bdd100k", "images", "100k", "val"),
    "Labels dir":   os.path.join(BDD_RAW_DIR, "bdd100k", "labels"),
}

# Also check common alternative paths
alt_checks = {
    "Train images (alt)": os.path.join(BDD_RAW_DIR, "images", "100k", "train"),
    "Val images (alt)":   os.path.join(BDD_RAW_DIR, "images", "100k", "val"),
    "Labels dir (alt)":   os.path.join(BDD_RAW_DIR, "labels"),
}

all_ok = True
for desc, path in checks.items():
    if not check_path(path, desc):
        # Try alternative path
        alt_desc = desc + " (alt)"
        if alt_desc in alt_checks:
            if not check_path(alt_checks[alt_desc], alt_desc):
                all_ok = False

if all_ok:
    print("\n🎯 All files verified! Proceed to notebook 02.")
else:
    print("\n⚠️ Some files are missing. Please download and extract first.")

## 7 — Inspect Detection Labels

In [ ]:
import json
import glob

# Find detection label files
label_candidates = [
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "det_20", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "det_20", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "det_train.json"),
    # BDD100K v1 format
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "bdd100k_labels_images_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "bdd100k_labels_images_train.json"),
]

train_label_path = None
for candidate in label_candidates:
    if os.path.isfile(candidate):
        train_label_path = candidate
        print(f"✅ Found train labels: {candidate}")
        break

if train_label_path is None:
    print("❌ Could not find train label file. Searching...")
    json_files = glob.glob(os.path.join(BDD_RAW_DIR, "**", "*.json"), recursive=True)
    for f in json_files[:10]:
        print(f"  Found: {f}")
    print("\n  ↑ One of these should be the detection label file.")
    print("  Update the path in notebook 02 accordingly.")

In [ ]:
# Preview a few label entries
if train_label_path:
    with open(train_label_path, 'r') as f:
        data = json.load(f)
    
    print(f"Total frames: {len(data)}")
    print(f"\nFirst entry keys: {list(data[0].keys())}")
    print(f"\nSample entry:")
    print(json.dumps(data[0], indent=2)[:1000])
    
    # Count categories
    categories = set()
    for frame in data[:1000]:  # Check first 1000
        for label in (frame.get('labels') or []):
            categories.add(label.get('category', 'unknown'))
    
    print(f"\nCategories found (first 1000 frames): {sorted(categories)}")

## 8 — (Optional) Quick Look at Lane Annotations

If you downloaded lane marking annotations, let's peek at them for future reference.
**We will NOT train on these yet.**

In [ ]:
# Search for lane-related files
lane_files = glob.glob(os.path.join(BDD_RAW_DIR, "**", "*lane*"), recursive=True)
lane_files += glob.glob(os.path.join(BDD_RAW_DIR, "**", "*drivable*"), recursive=True)

if lane_files:
    print("🔍 Lane / drivable area files found:")
    for f in lane_files[:10]:
        size_mb = os.path.getsize(f) / (1024**2) if os.path.isfile(f) else 0
        print(f"   {f}  ({size_mb:.1f} MB)")
    print("\n📌 These will be used in future lane marking notebooks.")
else:
    print("ℹ️ No lane marking files found yet. Download them later for lane work.")

## 9 — Save Configuration for Other Notebooks

In [ ]:
import yaml

# Save paths config so other notebooks can load it
config = {
    "ecocar_root": ECOCAR_ROOT,
    "bdd_raw_dir": BDD_RAW_DIR,
    "train_label_path": train_label_path,
}

config_path = os.path.join(ECOCAR_ROOT, "paths_config.yaml")
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Path config saved: {config_path}")
print(f"\n🎯 Dataset setup complete! Proceed to notebook 02 for BDD100K preparation.")

---

### Summary

| Downloaded | For | Status |
|---|---|---|
| `bdd100k_images_100k.zip` | Detection training | ✅ Required now |
| `bdd100k_labels_release.zip` | Detection labels | ✅ Required now |
| Lane / drivable annotations | Future lane work | ⏳ Optional now |

**Next:** Run `02_bdd100k_preparation.ipynb` to convert labels to YOLO format.